# 돌봄부담 I 블록(I10~I14) 응답 비교와 영향 문항 시각화**대상 데이터**: `ABC8pioneer3.2024_care_burden_std09` (2024 발달장애인 일과 삶 실태조사, 3,000가구 원본)**작성일**: 2026-08-31## 이 노트북의 목적1. **1부** — 원 문항 **I10~I14**의 응답을 여러 각도로 비교한다. 이 문항들은 target `I13`(돌봄부담)과   같은 설문 블록에 있어, 예측 모델에서 Data Leakage 위험으로 전면 배제된 변수군이다(FR-004b).   *왜 배제해야 했는지*를 눈으로 확인하는 것이 1부의 목적이다.2. **2부** — `I13` 분류에 영향을 크게 주는 문항을 찾아 시각화한다. I 블록을 포함했을 때와   제외했을 때가 어떻게 달라지는지 함께 본다.## 척도 방향 주의`I13`은 **1이 최고부담, 5가 부담 없음**이다. 숫자가 작을수록 부담이 크다.그래프의 색과 정렬은 모두 이 방향을 전제로 한다.## 자격증명에 관한 안내아래 접속 정보는 요청에 따라 노트북에 직접 넣었다. 외부에 공유하거나 공개 저장소에 올릴 때는`os.environ`으로 분리하는 편이 안전하다. 셀 안에 환경변수 대체 경로를 함께 넣어 두었다.

## 0. 준비

In [ ]:
# 필요한 패키지 설치 (이미 설치돼 있으면 건너뜁니다)%pip install -q pymysql pandas numpy matplotlib seaborn

In [ ]:
import os, math, warningsimport numpy as npimport pandas as pdimport pymysqlimport matplotlib as mplimport matplotlib.pyplot as pltimport seaborn as snswarnings.filterwarnings("ignore")# --- 한글 폰트 ---KFONT = next((c for c in ["AppleGothic", "Malgun Gothic", "NanumGothic", "NanumBarunGothic"]              if any(f.name == c for f in mpl.font_manager.fontManager.ttflist)), None)if KFONT:    mpl.rcParams["font.family"] = KFONTelse:    print("경고: 한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")mpl.rcParams["axes.unicode_minus"] = Falsempl.rcParams["figure.dpi"] = 110sns.set_theme(style="whitegrid", font=KFONT or "sans-serif")print("한글 폰트:", KFONT or "없음")

In [ ]:
# ── 데이터베이스 접속 정보 ─────────────────────────────────────────# 환경변수가 있으면 그 값을, 없으면 아래 기본값을 사용합니다.DB = dict(    host     = os.environ.get("CB_DB_HOST", "mis.iptime.org"),    port     = int(os.environ.get("CB_DB_PORT", 13306)),    user     = os.environ.get("CB_DB_USER", "pioneer3"),    password = os.environ.get("CB_DB_PASSWORD", "pioneer26"),    database = os.environ.get("CB_DB_NAME", "ABC8pioneer3"),    charset  = "utf8mb4",    connect_timeout = 20,)SRC_TABLE = "2024_care_burden_std09"   # 원본 테이블def connect():    return pymysql.connect(**DB)def q(sql, params=None):    con = connect()    try:        return pd.read_sql(sql, con, params=params)    finally:        con.close()print("접속 확인:", q("SELECT VERSION() AS v").iloc[0, 0])print("행 수:", q(f"SELECT COUNT(*) AS n FROM `{SRC_TABLE}`").iloc[0, 0])

In [ ]:
# 원본 테이블 전체 적재 (45컬럼 × 3,000행)df = q(f"SELECT * FROM `{SRC_TABLE}`")# TEXT 컬럼에 숫자 문자열과 공백(' ')이 섞여 있어 수치화한다.for c in df.columns:    if df[c].dtype == object:        df[c] = pd.to_numeric(df[c].astype(str).str.strip().replace("", np.nan), errors="coerce")# 설문지 척도 밖 무응답 코드 → 결측SENTINEL = {    "age_disability_suspected": [999],    "has_chronic_disease": [99],    "family_support_for_employment": [9],    "school_helpfulness": [9],    "daily_routine_satisfaction": [9],    "wanted_to_stay_at_last_job": [9],}for c, vals in SENTINEL.items():    if c in df.columns:        df[c] = df[c].replace(vals, np.nan)print(df.shape)df.head(3)

In [ ]:
# ── I 블록 문항 라벨 (2024 실태조사 설문지 원문 기준) ──────────────I_BLOCK = {    "care_burden": dict(code="I13", name="돌봄부담정도", kind="target",        q="보호자님이 당사자를 돌보거나 보호하면서 느끼는 전반적 부담정도",        labels={1:"매우 부담된다", 2:"부담되는 편", 3:"그저 그렇다",                4:"부담되지 않는 편", 5:"전혀 부담되지 않는다"}),    "care_difficulty_top1": dict(code="I10", name="돌봄어려움1순위", kind="nominal",        q="돌보거나 보호할 때 주로 겪고 있는 어려움 (1순위)",        labels={1:"보호자 직업활동 지장", 2:"보호자 육체적 피로·건강악화", 3:"보호자 정신적 스트레스",                4:"사회활동 지장", 5:"여가·휴식 제한", 6:"가족 간 다툼·불화",                7:"돌봐줄 사람·기관 없음", 8:"당사자 장애상태 악화", 9:"주변의 시선·편견",                10:"돌봄 비용 부담", 11:"당사자 미래 걱정", 12:"기타", 13:"특별히 없음"}),    "work_care_gap_exp": dict(code="I11", name="근로중돌봄공백경험", kind="binary",        q="일하면서 예정에 없던 일로 돌봄 제공자를 추가로 구해본 경험 (취업 보호자만 응답)",        labels={1:"경험 있음", 2:"경험 없음"}),    "work_care_gap_hours": dict(code="I11_H", name="월평균돌봄공백시간", kind="continuous",        q="추가로 구한 돌봄의 한 달 평균 시간", labels=None),    "integrated_care_awareness": dict(code="I12", name="통합돌봄제도인지도", kind="ordinal",        q="최중증 발달장애인 통합돌봄 제도 인지도",        labels={1:"구체적으로 앎", 2:"일부 앎", 3:"사업 취지만 앎", 4:"모름"}),    "needed_care_service_type": dict(code="I12_1", name="필요한통합돌봄서비스유형", kind="nominal",        q="가장 도움이 될 것으로 생각하는 통합돌봄 서비스 유형",        labels={1:"주간 그룹형 지원", 2:"주간 개별 지원", 3:"24시간 개별 지원", 4:"필요한 서비스 없음"}),    "caregiver_life_satisfaction": dict(code="I14", name="보호자삶만족도", kind="ordinal",        q="보호자 본인의 현재 삶에 대한 전반적 만족도",        labels={1:"매우 불만족", 2:"불만족하는 편", 3:"그저 그렇다",                4:"만족하는 편", 5:"매우 만족"}),}IB = list(I_BLOCK.keys())TARGET = "care_burden"BURDEN_LABEL = I_BLOCK[TARGET]["labels"]# 부담이 클수록 진한 빨강 (1=최고부담)BURDEN_COLORS = {1:"#8B1A1A", 2:"#C0392B", 3:"#E59866", 4:"#7FB3D5", 5:"#2874A6"}pd.DataFrame([{"컬럼":k, "문항":v["code"], "라벨":v["name"], "유형":v["kind"],               "결측":int(df[k].isna().sum())} for k,v in I_BLOCK.items()])

---# 1부. I10~I14 응답 비교같은 설문 블록 안에서 일곱 문항이 서로 어떻게 맞물려 있는지를 여덟 가지 방식으로 본다.

### 1.1 응답 분포 — 문항별 막대그래프각 문항이 어떤 값에 몰려 있는지부터 확인한다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))for ax, col in zip(axes.ravel(), [c for c in IB if I_BLOCK[c]["kind"] != "continuous"]):    info = I_BLOCK[col]    vc = df[col].value_counts().sort_index()    lab = [info["labels"].get(int(i), str(i)) for i in vc.index]    colors = [BURDEN_COLORS.get(int(i), "#5D6D7E") for i in vc.index] if col == TARGET else "#5D6D7E"    ax.barh(range(len(vc)), vc.values, color=colors)    ax.set_yticks(range(len(vc))); ax.set_yticklabels(lab, fontsize=8)    ax.invert_yaxis()    ax.set_title(f'{info["code"]} {info["name"]}  (결측 {df[col].isna().sum():,})', fontsize=10)    for i, v in enumerate(vc.values):        ax.text(v, i, f" {v:,}", va="center", fontsize=8)    ax.set_xlabel("")fig.suptitle("I 블록 문항별 응답 분포", fontsize=13, y=1.00)plt.tight_layout(); plt.show()

### 1.2 결측 구조 — 조건부 문항이 만드는 패턴I 블록의 결측은 데이터 누락이 아니라 **분기 문항의 "해당 없음"** 이다.`I11`은 취업 보호자만, `I11_H`는 그중 돌봄 공백 경험자만 답한다.

In [ ]:
miss = df[IB].isna().mean().sort_values(ascending=False) * 100fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios":[1, 1.4]})ax = axes[0]bars = ax.barh([f'{I_BLOCK[c]["code"]} {I_BLOCK[c]["name"]}' for c in miss.index],               miss.values, color=["#C0392B" if v > 50 else "#F39C12" if v > 10 else "#5DADE2" for v in miss.values])ax.set_xlabel("결측률 (%)"); ax.set_xlim(0, 100); ax.set_title("문항별 결측률")for b, v in zip(bars, miss.values):    ax.text(v + 1, b.get_y() + b.get_height()/2, f"{v:.1f}%", va="center", fontsize=9)# 결측 패턴 매트릭스 (행=응답자 표본, 열=문항)ax = axes[1]samp = df[IB].isna().astype(int).sample(400, random_state=0).sort_values(IB)sns.heatmap(samp.T, cmap=["#EAF2F8", "#34495E"], cbar=False, ax=ax,            yticklabels=[f'{I_BLOCK[c]["code"]}' for c in IB], xticklabels=False)ax.set_title("결측 패턴 (임의 400명 · 진한 색 = 결측)"); ax.set_xlabel("응답자")plt.tight_layout(); plt.show()print("I11(돌봄공백경험) 응답자 =", int(df["work_care_gap_exp"].notna().sum()), "명 — 취업 보호자만 응답")print("그중 I11_H(공백시간) 응답 =", int(df["work_care_gap_hours"].notna().sum()), "명 — 공백 경험자만 응답")

### 1.3 I13 × I14 교차 히트맵 — 왜 I14를 배제했는가`I14`(보호자 삶 만족도)는 target `I13`과 상호정보량이 38개 설명변수 중 가장 높았다.두 문항의 교차표를 보면 그 이유가 드러난다.

In [ ]:
ct  = pd.crosstab(df[TARGET], df["caregiver_life_satisfaction"])ctp = pd.crosstab(df[TARGET], df["caregiver_life_satisfaction"], normalize="index") * 100yl = [f'{i} {BURDEN_LABEL[i]}' for i in ct.index]xl = [f'{j} {I_BLOCK["caregiver_life_satisfaction"]["labels"][j]}' for j in ct.columns]fig, axes = plt.subplots(1, 2, figsize=(16, 5))sns.heatmap(ct, annot=True, fmt=",d", cmap="Reds", ax=axes[0],            xticklabels=xl, yticklabels=yl, cbar_kws={"label":"명"})axes[0].set_title("건수"); axes[0].set_xlabel("I14 보호자 삶 만족도"); axes[0].set_ylabel("I13 돌봄부담")sns.heatmap(ctp, annot=True, fmt=".1f", cmap="Reds", ax=axes[1],            xticklabels=xl, yticklabels=yl, cbar_kws={"label":"%"})axes[1].set_title("행 기준 비율 (%)"); axes[1].set_xlabel("I14 보호자 삶 만족도"); axes[1].set_ylabel("")fig.suptitle("I13 돌봄부담 × I14 보호자 삶 만족도 — 강한 대각 구조", fontsize=13, y=1.02)plt.tight_layout(); plt.show()print("두 문항의 순위상관 (Spearman):",      round(df[[TARGET, "caregiver_life_satisfaction"]].corr(method="spearman").iloc[0, 1], 3))

### 1.4 I13 단계별 I10 구성 — 부담 단계에 따라 어려움의 종류가 달라지는가

In [ ]:
sub = df[[TARGET, "care_difficulty_top1"]].dropna()ct  = pd.crosstab(sub[TARGET], sub["care_difficulty_top1"], normalize="index") * 100lab = I_BLOCK["care_difficulty_top1"]["labels"]ct.columns = [lab.get(int(c), str(c)) for c in ct.columns]keep = ct.mean().sort_values(ascending=False).head(8).indexct = ct[keep]ax = ct.plot(kind="barh", stacked=True, figsize=(14, 5), colormap="tab20", width=0.75)ax.set_yticklabels([f'{i} {BURDEN_LABEL[i]}' for i in ct.index])ax.invert_yaxis()ax.set_xlabel("구성비 (%)"); ax.set_ylabel("I13 돌봄부담"); ax.set_xlim(0, 100)ax.set_title("I13 단계별 I10 돌봄 어려움 1순위 구성 (상위 8개 항목)", fontsize=12)ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, title="I10 어려움")plt.tight_layout(); plt.show()display(ct.round(1))

### 1.5 I13별 I11 · I12 · I12_1 비교 — 세 문항을 한 화면에

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))for ax, col in zip(axes, ["work_care_gap_exp", "integrated_care_awareness", "needed_care_service_type"]):    info = I_BLOCK[col]    sub = df[[TARGET, col]].dropna()    ct = pd.crosstab(sub[TARGET], sub[col], normalize="index") * 100    ct.columns = [info["labels"].get(int(c), str(c)) for c in ct.columns]    ct.plot(kind="bar", ax=ax, colormap="viridis", width=0.8, edgecolor="white", linewidth=0.4)    ax.set_xticklabels([f"{i}단계" for i in ct.index], rotation=0)    ax.set_xlabel("I13 돌봄부담 (1=최고부담)"); ax.set_ylabel("비율 (%)")    ax.set_title(f'{info["code"]} {info["name"]}\n(응답 {len(sub):,}명)', fontsize=10)    ax.legend(fontsize=7, loc="upper right")fig.suptitle("I13 단계별 응답 구성 비교", fontsize=13, y=1.03)plt.tight_layout(); plt.show()

### 1.6 I11_H 돌봄 공백 시간 분포 — 표본이 매우 적은 문항응답자가 59명뿐이라 단계별 비교의 신뢰구간이 매우 넓다. 개별 점을 함께 찍어 표본 크기를 드러낸다.

In [ ]:
sub = df[[TARGET, "work_care_gap_hours"]].dropna()if len(sub) == 0:    print("응답자가 없습니다.")else:    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))    sns.boxplot(data=sub, x=TARGET, y="work_care_gap_hours", ax=axes[0],                palette=[BURDEN_COLORS[i] for i in sorted(sub[TARGET].unique())], width=0.55)    sns.stripplot(data=sub, x=TARGET, y="work_care_gap_hours", ax=axes[0],                  color="black", size=3.5, alpha=0.6, jitter=0.2)    axes[0].set_xlabel("I13 돌봄부담 (1=최고부담)"); axes[0].set_ylabel("월평균 돌봄공백 시간")    axes[0].set_title(f"I11_H 분포 (n={len(sub)})")    cnt = sub[TARGET].value_counts().sort_index()    axes[1].bar([f"{i}단계" for i in cnt.index], cnt.values,                color=[BURDEN_COLORS[i] for i in cnt.index])    for i, v in enumerate(cnt.values):        axes[1].text(i, v, f"{v}명", ha="center", va="bottom", fontsize=9)    axes[1].set_title("단계별 응답자 수 — 표본이 매우 적음"); axes[1].set_ylabel("명")    plt.tight_layout(); plt.show()

### 1.7 I 블록 내부 연관성 — Cramér's V 행렬일곱 문항이 서로 얼마나 얽혀 있는지 본다. 값이 클수록 두 문항이 같은 것을 재고 있다는 뜻이다.

In [ ]:
def cramers_v(a, b):    """두 범주형 변수의 Cramér's V (bias 보정)."""    s = pd.DataFrame({"a": a, "b": b}).dropna()    if s["a"].nunique() < 2 or s["b"].nunique() < 2:        return np.nan    ct = pd.crosstab(s["a"], s["b"]).values    n  = ct.sum()    exp = np.outer(ct.sum(1), ct.sum(0)) / n    chi2 = ((ct - exp) ** 2 / np.where(exp == 0, np.nan, exp)).sum()    phi2 = chi2 / n    r, k = ct.shape    phi2c = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))    rc = r - (r - 1) ** 2 / (n - 1)    kc = k - (k - 1) ** 2 / (n - 1)    d = min(kc - 1, rc - 1)    return np.sqrt(phi2c / d) if d > 0 else np.nancols = [c for c in IB if I_BLOCK[c]["kind"] != "continuous"]V = pd.DataFrame(index=cols, columns=cols, dtype=float)for i in cols:    for j in cols:        V.loc[i, j] = 1.0 if i == j else cramers_v(df[i], df[j])names = [f'{I_BLOCK[c]["code"]}\n{I_BLOCK[c]["name"]}' for c in cols]plt.figure(figsize=(8.5, 7))mask = np.triu(np.ones_like(V, dtype=bool), k=1)sns.heatmap(V.astype(float), mask=mask, annot=True, fmt=".3f", cmap="RdPu",            xticklabels=names, yticklabels=names, vmin=0, vmax=0.5,            cbar_kws={"label":"Cramér's V"}, linewidths=0.5)plt.title("I 블록 문항 간 연관성", fontsize=12)plt.xticks(fontsize=8); plt.yticks(fontsize=8, rotation=0)plt.tight_layout(); plt.show()print("target(I13)과의 연관성 순위")print(V[TARGET].drop(TARGET).sort_values(ascending=False).round(3).to_string())

### 1.8 I13 × I14 버블 차트 — 응답이 실제로 몰려 있는 자리히트맵과 같은 정보를 원 크기로 보면 표본이 어디에 집중돼 있는지가 더 잘 보인다.

In [ ]:
sub = df[[TARGET, "caregiver_life_satisfaction"]].dropna()g = sub.groupby([TARGET, "caregiver_life_satisfaction"]).size().reset_index(name="n")plt.figure(figsize=(8.5, 6))plt.scatter(g["caregiver_life_satisfaction"], g[TARGET], s=g["n"] * 1.6,            c=[BURDEN_COLORS[int(i)] for i in g[TARGET]], alpha=0.75, edgecolors="white", linewidth=1.2)for _, r in g.iterrows():    if r["n"] >= 60:        plt.text(r["caregiver_life_satisfaction"], r[TARGET], int(r["n"]),                 ha="center", va="center", fontsize=8, color="white", fontweight="bold")plt.xticks([1,2,3,4,5], [f'{i}\n{I_BLOCK["caregiver_life_satisfaction"]["labels"][i]}' for i in range(1,6)], fontsize=8)plt.yticks([1,2,3,4,5], [f'{i} {BURDEN_LABEL[i]}' for i in range(1,6)], fontsize=8)plt.gca().invert_yaxis()plt.xlabel("I14 보호자 삶 만족도"); plt.ylabel("I13 돌봄부담")plt.title("응답 집중도 — 원 크기 = 응답자 수", fontsize=12)plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---# 2부. I13 분류에 영향을 크게 주는 문항여기서부터는 I 블록을 넘어 **44개 설명변수 전체**를 대상으로, `I13` 분류에 어떤 문항이얼마나 기여하는지를 본다.

In [ ]:
# ── 영향도 지표 두 가지 ────────────────────────────────────────────def entropy(v):    p = pd.Series(v).value_counts(normalize=True)    return float(-(p * np.log(p)).sum())def mi_ratio(x, y):    """정규화 상호정보량 MI/H(y). 결측은 하나의 범주로 취급한다."""    s = pd.DataFrame({"x": pd.Series(x).fillna("__NA__"), "y": y}).dropna(subset=["y"])    Hy = entropy(s["y"]); n = len(s)    if Hy == 0: return 0.0    cond = sum(len(g) / n * entropy(g["y"]) for _, g in s.groupby("x"))    return (Hy - cond) / Hy * 100TARGET_CODE = "I13"LEAK = ["care_difficulty_top1", "work_care_gap_exp", "work_care_gap_hours",        "integrated_care_awareness", "needed_care_service_type", "caregiver_life_satisfaction"]FEATS = [c for c in df.columns if c != TARGET]imp = pd.DataFrame({    "feature": FEATS,    "MI":      [mi_ratio(df[c], df[TARGET]) for c in FEATS],    "CramersV":[cramers_v(df[c].fillna("__NA__"), df[TARGET]) for c in FEATS],})imp["leakage"] = imp["feature"].isin(LEAK)imp = imp.sort_values("MI", ascending=False).reset_index(drop=True)imp.head(12).round(3)

### 2.1 44개 문항 영향도 순위 — Leakage 변수를 색으로 구분빨간 막대가 target과 같은 설문 블록에 있어 배제된 변수다.

In [ ]:
top = imp.head(25).iloc[::-1]plt.figure(figsize=(11, 8))plt.barh(range(len(top)), top["MI"],         color=["#C0392B" if l else "#5DADE2" for l in top["leakage"]])plt.yticks(range(len(top)), top["feature"], fontsize=8.5)for i, (v, l) in enumerate(zip(top["MI"], top["leakage"])):    plt.text(v + 0.12, i, f"{v:.2f}%" + ("  ← 배제" if l else ""), va="center", fontsize=8,             color="#C0392B" if l else "#34495E")plt.xlabel("MI / H(I13)  (%)"); plt.title("I13 설명력 상위 25개 문항", fontsize=12)plt.tight_layout(); plt.show()print("Leakage 변수가 상위 10위 안에 든 개수:", int(imp.head(10)["leakage"].sum()))

### 2.2 Leakage 배제 전후 비교 — 무엇이 사라지고 무엇이 올라오는가

In [ ]:
before = imp.head(10)after  = imp[~imp["leakage"]].head(10)fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), sharex=True)for ax, d, t in [(axes[0], before.iloc[::-1], "배제 전 — 전체 44변수"),                 (axes[1], after.iloc[::-1],  "배제 후 — 38변수 (실제 모델 입력)")]:    ax.barh(range(len(d)), d["MI"], color=["#C0392B" if l else "#5DADE2" for l in d["leakage"]])    ax.set_yticks(range(len(d))); ax.set_yticklabels(d["feature"], fontsize=8.5)    for i, v in enumerate(d["MI"]):        ax.text(v + 0.15, i, f"{v:.2f}%", va="center", fontsize=8)    ax.set_title(t, fontsize=11); ax.set_xlabel("MI / H(I13)  (%)")fig.suptitle("Leakage 변수 배제가 설명력 구조에 미치는 영향", fontsize=13, y=1.02)plt.tight_layout(); plt.show()print(f"최상위 설명력  배제 전 {imp.iloc[0]['MI']:.2f}%  →  배제 후 {after.iloc[0]['MI']:.2f}%")

### 2.3 두 지표의 일치도 — MI와 Cramér's V상호정보량은 정보량 관점, Cramér's V는 연관 강도 관점이다. 두 지표가 크게 어긋나는 변수는레벨 수가 많아 과대평가된 경우일 수 있으므로 따로 확인한다.

In [ ]:
plt.figure(figsize=(9, 7))for leak, g in imp.groupby("leakage"):    plt.scatter(g["CramersV"], g["MI"], s=70, alpha=0.8,                c="#C0392B" if leak else "#5DADE2",                label="배제 (I 블록)" if leak else "사용 (38변수)", edgecolors="white")for _, r in imp.head(14).iterrows():    plt.annotate(r["feature"], (r["CramersV"], r["MI"]), fontsize=7.5,                 xytext=(4, 3), textcoords="offset points")plt.xlabel("Cramér's V"); plt.ylabel("MI / H(I13)  (%)")plt.title("영향도 지표 비교 — 두 지표가 대체로 일치한다", fontsize=12)plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()print("두 지표의 순위상관 (Spearman):",      round(imp[["MI", "CramersV"]].corr(method="spearman").iloc[0, 1], 3))

### 2.4 상위 문항별 I13 분포 — 값이 바뀌면 부담 구성이 어떻게 달라지는가배제 후 상위 6개 문항 각각에 대해, 응답값별 I13 구성비를 100% 누적 막대로 그린다.막대 안의 진한 빨강이 넓을수록 그 응답을 한 집단에 고부담이 많다는 뜻이다.

In [ ]:
top6 = imp[~imp["leakage"]].head(6)["feature"].tolist()SCALE_NOTE = {    "help_needed_hours": "1 일과 대부분(12h+) → 5 필요없음",    "understands_work_meaning": "1 잘 이해 → 4 전혀 못함",    "can_work_standard_job": "1 통상근로 가능 → 4 집안일도 불가",    "caregiver_age": "주 보호자 만 나이",    "age_disability_suspected": "장애 의심 시작 나이",    "severe_dd_job_willingness": "1 예, 2 아니오",}fig, axes = plt.subplots(2, 3, figsize=(17, 9))for ax, col in zip(axes.ravel(), top6):    s = df[[col, TARGET]].dropna()    if s[col].nunique() > 8:                       # 연속형은 구간화        s[col] = pd.qcut(s[col], 5, duplicates="drop")    ct = pd.crosstab(s[col], s[TARGET], normalize="index") * 100    ct = ct.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)    ct.plot(kind="bar", stacked=True, ax=ax, width=0.8,            color=[BURDEN_COLORS[c] for c in ct.columns], legend=False, edgecolor="white", linewidth=0.4)    ax.set_title(f'{col}\n{SCALE_NOTE.get(col, "")}  ·  MI {imp.set_index("feature").loc[col,"MI"]:.2f}%', fontsize=9.5)    ax.set_xlabel(""); ax.set_ylabel("I13 구성비 (%)"); ax.set_ylim(0, 100)    ax.tick_params(axis="x", rotation=30, labelsize=7.5)handles = [plt.Rectangle((0,0),1,1, color=BURDEN_COLORS[i]) for i in range(1,6)]fig.legend(handles, [f"{i} {BURDEN_LABEL[i]}" for i in range(1,6)],           loc="lower center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.02))fig.suptitle("상위 6개 문항의 응답값별 I13 구성 (Leakage 배제 후)", fontsize=13, y=1.00)plt.tight_layout(); plt.show()

### 2.5 고부담군 비율로 본 단조성 — 순서형 문항이 실제로 단조적인가I13을 **고부담군(1~2단계) 여부**로 이분화하고, 각 문항의 응답값별 고부담 비율을 선으로 잇는다.선이 한 방향으로 기울면 그 문항은 부담과 단조 관계를 갖는다.

In [ ]:
ORD = ["help_needed_hours", "understands_work_meaning", "can_work_standard_job",       "overall_health", "job_ability_mobility", "family_support_for_employment",       "daily_routine_satisfaction", "final_school"]ORD = [c for c in ORD if c in df.columns]df["_high"] = (df[TARGET] <= 2).astype(int)      # 1~2단계 = 고부담군base = df["_high"].mean() * 100plt.figure(figsize=(12, 6))for col in ORD:    s = df[[col, "_high"]].dropna()    g = s.groupby(col)["_high"].agg(["mean", "size"])    g = g[g["size"] >= 30]    plt.plot(range(len(g)), g["mean"] * 100, marker="o", label=col, linewidth=1.8, markersize=5)plt.axhline(base, color="gray", linestyle="--", linewidth=1)plt.text(0, base + 0.8, f"전체 평균 {base:.1f}%", fontsize=8, color="gray")plt.xlabel("응답값 순서 (왼쪽 = 낮은 코드)"); plt.ylabel("고부담군(1~2단계) 비율 (%)")plt.title("순서형 문항의 단조성 점검 — 표본 30건 이상 구간만", fontsize=12)plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3)plt.tight_layout(); plt.show()df.drop(columns="_high", inplace=True)

### 2.6 누적 설명력 곡선 — 몇 개 문항이면 충분한가문항을 설명력 순으로 더해갈 때 누적 MI가 어떻게 늘어나는지 본다.상호정보량은 가법이 아니므로 **상한의 눈금**으로만 읽어야 하며, 실제 문항 수는38변수 전체 모델 대비 성능 손실 기준(FR-004c)으로 정해야 한다.

In [ ]:
use = imp[~imp["leakage"]].reset_index(drop=True)cum = use["MI"].cumsum() / use["MI"].sum() * 100fig, ax1 = plt.subplots(figsize=(12, 5.5))ax1.bar(range(len(use)), use["MI"], color="#5DADE2", alpha=0.85)ax1.set_ylabel("개별 MI / H  (%)", color="#2874A6"); ax1.set_xlabel("문항 순위")ax1.tick_params(axis="y", labelcolor="#2874A6")ax2 = ax1.twinx()ax2.plot(range(len(use)), cum, color="#C0392B", marker="o", markersize=3.5, linewidth=1.8)ax2.set_ylabel("누적 비중 (%)", color="#C0392B"); ax2.tick_params(axis="y", labelcolor="#C0392B")for th in (80, 90):    k = int((cum >= th).idxmax()) + 1    ax2.axhline(th, color="gray", linestyle=":", linewidth=1)    ax2.annotate(f"{th}% 도달: 상위 {k}문항", (k - 1, th), fontsize=8.5,                 xytext=(6, -14), textcoords="offset points", color="#7B241C")ax1.set_xticks(range(0, len(use), 2))ax1.set_xticklabels(use["feature"][::2], rotation=75, fontsize=7)plt.title("문항 수에 따른 누적 설명력 (38변수 기준)", fontsize=12)plt.tight_layout(); plt.show()print(f"하위 {(use['MI'] < 1).sum()}개 문항의 개별 설명력이 1% 미만")

---## 3. 정리### 1부에서 확인한 것- `I14`(보호자 삶 만족도)는 `I13`과 뚜렷한 대각 구조를 이룬다. 두 문항 모두 **보호자 자신의  주관적 상태**를 5점 척도로 묻기 때문이다. 이 변수를 모델에 넣으면 "삶이 불만족스러우면  돌봄부담이 크다"는 동어반복을 학습하게 되고, 문항을 줄인 진단 도구의 존재 이유가 사라진다.- `I10`(돌봄 어려움)과 `I12_1`(필요한 서비스)은 **부담이 이미 있음을 전제**하고 묻는 문항이다.  부담의 원인이 아니라 결과에 가깝다.- `I11`·`I11_H`는 취업 보호자, 그중에서도 돌봄 공백 경험자만 답해 표본이 급격히 줄어든다.이 세 가지가 I 블록 6개 변수를 전면 배제한 근거다(FR-004b).### 2부에서 확인한 것- 배제 전 설명력 1위는 `I14`였고, 배제 후 1위는 `help_needed_hours`(G6, 도움 필요 시간)로 바뀐다.  최상위 설명력 자체가 크게 낮아지는데, 이것이 leakage 배제의 비용이다.- 남은 38개 문항의 개별 설명력은 전반적으로 약하다. 단일 문항으로 부담을 가르는 변수는 없고,  여러 문항의 조합이 필요하다.- 상호정보량은 **변수 하나씩만** 보는 지표라 조합 효과와 중복을 반영하지 못한다.  최종 문항 집합은 이 순위가 아니라 **38변수 전체 모델 대비 성능 손실 기준**으로 도출해야 한다(FR-004c).### 관련 문서- `docs/학습데이터셋-컬럼정의.md` — 38개 변수의 한글 라벨, 응답 척도와 방향, 도입 근거- `specs/001-care-burden-map/spec.md` — FR-004b(Leakage 배제), FR-004c(문항 집합 도출)- 학습용 데이터셋: `cb_dataset_v1` · `v_cb_tree_v1` · `v_cb_linear_v1` (split·cv_fold 고정)